# 03 — Turn a QA dataset into decision-model data
Core notebook. We use two **free, public** datasets:
- **PubMedQA** — yes/no/maybe biomedical questions → Noul + Choice + Score.
- **Banking77** — 77-way intent classification → Choice with hard negatives + Noul checks.

Output is JSONL in the schema documented in `data/SCHEMA.md`.


In [ ]:
from decision_qa import load_examples

pubmed = load_examples('pubmedqa', split='train', limit=8)
banking = load_examples('banking77', split='train', limit=8)
print('PubMedQA:', len(pubmed), '| Banking77:', len(banking))
print(pubmed[0].question, '->', pubmed[0].label)
print(banking[0].context, '->', banking[0].label)


## Conversion
For PubMedQA, the declared `final_decision` becomes a 3-way Choice, the `long_answer` a Noul statement (paired with a contradicted answer), and a 3-level Score rubric.
For Banking77, the gold intent is mixed with lexical **hard negatives** and the options are shuffled.


In [ ]:
from decision_qa import ConversionConfig, convert_pubmedqa, convert_banking77

cfg = ConversionConfig(n_distractors=3, seed=42)
pm_records = convert_pubmedqa(pubmed, cfg, split='train')
bk_records = convert_banking77(banking, cfg, split='train')
print(pm_records[0].model_dump_json(indent=2)[:900])


## Write and validate
`build_dataset` loads, converts, splits and writes the JSONL. We then assert the invariants: gold indices are in range, distractors never contain the gold label, and Noul pairs have both polarities.


In [ ]:
from decision_qa import build_dataset, read_jsonl

path = build_dataset('pubmedqa', 'data/pubmedqa_decisions.jsonl',
                     limits={'train': 40, 'val': 8, 'test': 8}, cfg=cfg)
records = read_jsonl(path)
print('records:', len(records), '| path:', path)

from decision_qa.questions import ChoiceQuestion, NoulQuestion
for r in records:
    for q in r.questions:
        if isinstance(q, ChoiceQuestion):
            assert 0 <= q.gold_index < len(q.options)
        if isinstance(q, NoulQuestion):
            assert q.statement  # both polarities are emitted per record
print('schema invariants OK')


Next: train on this data (04), evaluate it (05). Schema details: `data/SCHEMA.md`.
